# Day 2 - Data Engineering

This notebook covers loading raw Kaggle datasets (Train **and** Test), preprocessing,
combining, feature engineering, class imbalance mapping, and exporting two final
structured Delta tables — one for training and one for the pre-split holdout test set.

> **Change from original:** The pipeline now also ingests the four Test CSVs
> (`Test-*.csv`, `Test_Beneficiary*`, `Test_Inpatient*`, `Test_Outpatient*`) and saves
> a separate `capstone.bronze.claims_engineered_test` Delta table so Day 3 can use
> your real holdout instead of a random 80/20 split.

In [0]:
import os
import glob as _glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, count, isnan, isnull, sum as _sum,
    avg, round as _round, datediff, to_date, desc, countDistinct,
    month, year, max as _max, min as _min
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

VOLUME_PATH = "/Volumes/capstone/datasets/datasets/"


def find_file(keyword, prefix="Train"):
    """
    Uses os.listdir + startswith for exact prefix matching.
    Avoids glob unreliability on Databricks Unity Catalog FUSE mounts.
    """
    matches = [
        os.path.join(VOLUME_PATH, f)
        for f in os.listdir(VOLUME_PATH)
        if f.startswith(prefix) and keyword in f
    ]
    assert matches, f"No file found for prefix='{prefix}' keyword='{keyword}' in {VOLUME_PATH}"
    assert len(matches) == 1, f"Ambiguous match for '{keyword}': {matches}"
    return matches[0]


# ── Train file paths ──────────────────────────────────────────────────────────
TRAIN_INPATIENT_PATH   = find_file("Inpatientdata",  prefix="Train_")
TRAIN_OUTPATIENT_PATH  = find_file("Outpatientdata", prefix="Train_")
TRAIN_BENEFICIARY_PATH = find_file("Beneficiarydata",prefix="Train_")
TRAIN_LABEL_PATH       = find_file("",               prefix="Train-")

# ── Test file paths (NEW) ─────────────────────────────────────────────────────
TEST_INPATIENT_PATH    = find_file("Inpatientdata",  prefix="Test_")
TEST_OUTPATIENT_PATH   = find_file("Outpatientdata", prefix="Test_")
TEST_BENEFICIARY_PATH  = find_file("Beneficiarydata",prefix="Test_")
TEST_LABEL_PATH        = find_file("",               prefix="Test-")  # Provider + PotentialFraud labels

print("Resolved file paths:")
for label, p in [
    ("Train Inpatient",    TRAIN_INPATIENT_PATH),
    ("Train Outpatient",   TRAIN_OUTPATIENT_PATH),
    ("Train Beneficiary",  TRAIN_BENEFICIARY_PATH),
    ("Train Labels",       TRAIN_LABEL_PATH),
    ("Test  Inpatient",    TEST_INPATIENT_PATH),
    ("Test  Outpatient",   TEST_OUTPATIENT_PATH),
    ("Test  Beneficiary",  TEST_BENEFICIARY_PATH),
    ("Test  Labels",       TEST_LABEL_PATH),
]:
    print(f"  {label:20s} → {os.path.basename(p)}")

# ── Load Train DataFrames ─────────────────────────────────────────────────────
train_inpatient_df   = spark.read.csv(TRAIN_INPATIENT_PATH,   header=True, inferSchema=True, nullValue="NA")
train_outpatient_df  = spark.read.csv(TRAIN_OUTPATIENT_PATH,  header=True, inferSchema=True, nullValue="NA")
train_beneficiary_df = spark.read.csv(TRAIN_BENEFICIARY_PATH, header=True, inferSchema=True, nullValue="NA")
train_label_df       = spark.read.csv(TRAIN_LABEL_PATH,       header=True, inferSchema=True, nullValue="NA")

# ── Load Test DataFrames (NEW) ────────────────────────────────────────────────
test_inpatient_df    = spark.read.csv(TEST_INPATIENT_PATH,    header=True, inferSchema=True, nullValue="NA")
test_outpatient_df   = spark.read.csv(TEST_OUTPATIENT_PATH,   header=True, inferSchema=True, nullValue="NA")
test_beneficiary_df  = spark.read.csv(TEST_BENEFICIARY_PATH,  header=True, inferSchema=True, nullValue="NA")
test_label_df        = spark.read.csv(TEST_LABEL_PATH,        header=True, inferSchema=True, nullValue="NA")

print("\nAll datasets loaded successfully.")


## Pipeline Function

All preprocessing steps are wrapped in `build_claims_df()` so the exact same
transformations are applied to both the train and test datasets — guaranteeing
no feature-engineering drift between splits.

In [0]:
def recode_labels(label_df):
    """Recode PotentialFraud from Yes/No strings to 1/0 integers."""
    if "PotentialFraud" not in label_df.columns:
        print("WARNING: PotentialFraud not found — adding null column (unlabelled test set).")
        return label_df.withColumn("PotentialFraud", F.lit(None).cast("int"))
    return label_df.withColumn(
        "PotentialFraud",
        F.when(F.col("PotentialFraud") == "Yes", 1).otherwise(0)
    )


def recode_beneficiary(beneficiary_df):
    """Recode chronic condition columns and RenalDiseaseIndicator."""
    # ChronicCond_* from 1/2 to 1/0 binary flags
    chronic_cols = [c for c in beneficiary_df.columns if c.startswith("ChronicCond_")]
    for c in chronic_cols:
        beneficiary_df = beneficiary_df.withColumn(
            c,
            F.when(F.col(c) == 1, 1).when(F.col(c) == 2, 0).otherwise(None)
        )
    # RenalDiseaseIndicator from "Y"/"0" to 1/0
    beneficiary_df = beneficiary_df.withColumn(
        "RenalDiseaseIndicator",
        F.when(F.col("RenalDiseaseIndicator") == "Y", 1).otherwise(0)
    ).withColumnRenamed("RenalDiseaseIndicator", "HasRenalDisease")
    return beneficiary_df, chronic_cols


def build_claims_df(label_df, beneficiary_df, inpatient_df, outpatient_df):
    """
    Full Day-2 pipeline: recode → union → join → clean → impute →
    feature engineer → class weight → drop columns.

    Parameters
    ----------
    label_df       : DataFrame  Provider-level fraud labels (PotentialFraud Yes/No)
    beneficiary_df : DataFrame  Beneficiary demographic data
    inpatient_df   : DataFrame  Inpatient claims
    outpatient_df  : DataFrame  Outpatient claims

    Returns
    -------
    claims_df : DataFrame  Fully engineered, column-pruned claim DataFrame
    """

    # ── Step 1: Recode labels & beneficiary columns ───────────────────────────
    label_df = recode_labels(label_df)
    beneficiary_df, chronic_cols = recode_beneficiary(beneficiary_df)

    # ── Step 2: Add claim_type flag and union ─────────────────────────────────
    inpatient_df  = inpatient_df.withColumn("claim_type",  F.lit(1))
    outpatient_df = outpatient_df.withColumn("claim_type", F.lit(0))
    claims_df = inpatient_df.unionByName(outpatient_df, allowMissingColumns=True)

    # ── Step 3: Join Beneficiary and Labels ───────────────────────────────────
    claims_df = claims_df.join(beneficiary_df, on="BeneID",   how="left")
    claims_df = claims_df.join(label_df,       on="Provider", how="left")
    print(f"  Master claim DataFrame shape: {claims_df.count():,} rows × {len(claims_df.columns)} cols")

    # ── Step 4: Clean "NA" / empty strings to null ────────────────────────────
    for c, dtype in claims_df.dtypes:
        if dtype == "string":
            claims_df = claims_df.withColumn(
                c,
                F.when((F.col(c) == "NA") | (F.col(c) == ""), None).otherwise(F.col(c))
            )
    claims_df = claims_df.cache()
    print(f"  Cleaned & cached: {claims_df.count():,} rows")

    # ── Step 5: Impute DeductibleAmtPaid (inpatient median) ──────────────────
    median_deductible = claims_df.filter(F.col("claim_type") == 1) \
        .approxQuantile("DeductibleAmtPaid", [0.5], 0.01)[0]
    claims_df = claims_df.withColumn(
        "DeductibleAmtPaid",
        F.when((F.col("claim_type") == 1) & F.col("DeductibleAmtPaid").isNull(),
               median_deductible)
         .otherwise(F.col("DeductibleAmtPaid"))
    )

    # Fill missing ClmDiagnosisCode_1 with UNKNOWN
    claims_df = claims_df.withColumn(
        "ClmDiagnosisCode_1",
        F.when(F.col("ClmDiagnosisCode_1").isNull(), "UNKNOWN")
         .otherwise(F.col("ClmDiagnosisCode_1"))
    )

    # ── Step 6: Feature engineering ──────────────────────────────────────────
    diag_cols = [f"ClmDiagnosisCode_{i}" for i in range(1, 10)]
    proc_cols = [f"ClmProcedureCode_{i}" for i in range(1, 7)]
    phys_cols = ["AttendingPhysician", "OperatingPhysician", "OtherPhysician"]

    claims_df = claims_df.withColumn(
        "DiagnosisCodeCount",
        sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)
            for c in diag_cols if c in claims_df.columns)
    )
    claims_df = claims_df.withColumn(
        "ProcedureCodeCount",
        sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)
            for c in proc_cols if c in claims_df.columns)
    )
    claims_df = claims_df.withColumn(
        "PhysicianCount",
        sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in phys_cols)
    )
    claims_df = claims_df.withColumn(
        "HasOperatingPhysician",
        F.when(F.col("OperatingPhysician").isNotNull(), 1).otherwise(0)
    )
    claims_df = claims_df.withColumn(
        "HasOtherPhysician",
        F.when(F.col("OtherPhysician").isNotNull(), 1).otherwise(0)
    )

    for dt_col, fmt in [("ClaimStartDt", "yyyy-MM-dd"), ("ClaimEndDt", "yyyy-MM-dd"),
                         ("AdmissionDt",  "yyyy-MM-dd"), ("DischargeDt",  "yyyy-MM-dd"),
                         ("DOB",           "yyyy-MM-dd"), ("DOD",           "yyyy-MM-dd")]:
        claims_df = claims_df.withColumn(dt_col, F.to_date(F.col(dt_col), fmt))

    claims_df = claims_df.withColumn("ClaimDuration",
        F.datediff(F.col("ClaimEndDt"),  F.col("ClaimStartDt")))
    claims_df = claims_df.withColumn("LengthOfStay",
        F.datediff(F.col("DischargeDt"), F.col("AdmissionDt")))
    claims_df = claims_df.withColumn("Age",
        F.datediff(F.col("ClaimStartDt"), F.col("DOB")) / 365.25)
    claims_df = claims_df.withColumn("IsDeceased",
        F.when(F.col("DOD").isNotNull(), 1).otherwise(0))
    claims_df = claims_df.withColumn("ChronicConditionCount",
        sum(F.col(c) for c in chronic_cols))
    claims_df = claims_df.withColumn("HasPrimaryDiagnosis",
        F.when(F.col("ClmDiagnosisCode_1") != "UNKNOWN", 1).otherwise(0))
    claims_df = claims_df.withColumn("ICD_Chapter",
        F.substring("ClmDiagnosisCode_1", 1, 3))
    claims_df = claims_df.withColumn("MedicareCoverageMonths",
        F.col("NoOfMonths_PartACov") + F.col("NoOfMonths_PartBCov"))
    claims_df = claims_df.withColumn("IPtoOPReimbursementRatio",
        F.col("IPAnnualReimbursementAmt") / (F.col("OPAnnualReimbursementAmt") + 1))

    # ── Step 7: Class weights (skipped for unlabelled test sets) ────────────
    labelled_count = claims_df.filter(F.col("PotentialFraud").isNotNull()).count()
    if labelled_count == 0:
        claims_df = claims_df.withColumn("class_weight", F.lit(1.0))
        print("Skipping class weights — unlabelled test set.")
    else:
        fraud_count     = claims_df.filter(F.col("PotentialFraud") == 1).count()
        non_fraud_count = labelled_count - fraud_count
        weight_fraud     = labelled_count / (2.0 * fraud_count)     if fraud_count     > 0 else 1.0
        weight_non_fraud = labelled_count / (2.0 * non_fraud_count) if non_fraud_count > 0 else 1.0
        claims_df = claims_df.withColumn(
            "class_weight",
            F.when(F.col("PotentialFraud") == 1, weight_fraud).otherwise(weight_non_fraud)
        )
    # ── Step 8: Drop non-predictive columns ──────────────────────────────────
    cols_to_drop = [
        "AttendingPhysician", "OperatingPhysician", "OtherPhysician",
        "DOB", "DOD", "DiagnosisGroupCode", "ClmAdmitDiagnosisCode",
        "ClmDiagnosisCode_10", "County", "IPAnnualDeductibleAmt",
        "OPAnnualDeductibleAmt", "NoOfMonths_PartACov", "NoOfMonths_PartBCov",
        "IPAnnualReimbursementAmt", "OPAnnualReimbursementAmt"
    ]
    cols_to_drop.extend(proc_cols)
    cols_to_drop.extend(diag_cols)
    cols_to_drop = [c for c in cols_to_drop if c in claims_df.columns]
    claims_df = claims_df.drop(*cols_to_drop)

    return claims_df


print("build_claims_df() defined — ready to run on train and test sets.")


## Run Pipeline on Train Set

In [0]:
print("=" * 55)
print("Building TRAIN claims DataFrame...")
print("=" * 55)
train_claims_df = build_claims_df(
    label_df       = train_label_df,
    beneficiary_df = train_beneficiary_df,
    inpatient_df   = train_inpatient_df,
    outpatient_df  = train_outpatient_df,
)
print(f"\nTrain schema ({len(train_claims_df.columns)} cols):")
train_claims_df.printSchema()


## Run Pipeline on Test Set

In [0]:
print("=" * 55)
print("Building TEST claims DataFrame...")
print("=" * 55)
test_claims_df = build_claims_df(
    label_df       = test_label_df,
    beneficiary_df = test_beneficiary_df,
    inpatient_df   = test_inpatient_df,
    outpatient_df  = test_outpatient_df,
)
print(f"\nTest schema ({len(test_claims_df.columns)} cols):")
test_claims_df.printSchema()


## Save Both Delta Tables

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS capstone.bronze")

# ── Save Train ────────────────────────────────────────────────────────────────
TRAIN_TABLE = "capstone.bronze.claims_engineered"
train_claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TRAIN_TABLE)
print(f"✅ Train Delta table written : {TRAIN_TABLE}")
print(f"   Rows : {train_claims_df.count():,}")
print(f"   Cols : {len(train_claims_df.columns)}")

# ── Save Test (NEW) ───────────────────────────────────────────────────────────
TEST_TABLE = "capstone.bronze.claims_engineered_test"
test_claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TEST_TABLE)
print(f"\n✅ Test  Delta table written : {TEST_TABLE}")
print(f"   Rows : {test_claims_df.count():,}")
print(f"   Cols : {len(test_claims_df.columns)}")


In [0]:
print("\nTrain sample:")
train_claims_df.show(5)
print("\nTest sample:")
test_claims_df.show(5)
